# Dataset metrics: regional delta and alpha

This notebook is one of 5 focused notebooks split out of the original `esmrmb_variant_comparison.ipynb` (archived under `_archive/`) so each analysis stage can be opened and run on its own. No analysis or plotting logic changed in the split -- every cell below is copied verbatim from the original notebook; only the output path gained a `resample-ogse_rest_rician` segment naming the resampling model used (`build_resampled_contrasts`, fit with `M_ogse_rest_rician`), so a future alternative resampling model can sit alongside this one without collisions.

The sibling notebooks are: `signal_and_contrast_panels.ipynb`, `contrast_vs_lcf_panels.ipynb`, `dataset_metrics.ipynb`, `contrast_models_overlap.ipynb`, `cross_dataset_metrics.ipynb`.

## 1. Imports, provenance, and data model

Run the notebook from any directory inside the project. The helper below finds the project root without relying on a user-specific absolute path. All scientific calculations are implemented in `src/presentation_analysis/abstract_figures.py`; the notebook supplies configuration, displays intermediate QC, and exports reproducible artifacts.

The long master table is the source of truth. A signal row is identified by acquisition metadata (`subj`, `sheet`, $T_D$, $N$, source file and gradient step), spatial metadata (`roi`, rotated `direction`), statistic (`avg` or `std`), and processing provenance (`variant`, `analysis_tag`, `row_kind`). `signal_rotated` means the diffusion tensor and signal directions have already been expressed relative to the local ROI/fiber frame. The analysis never infers missing rows and never merges subjects or acquisition sheets.

Main units used below: $T_D$, $\tau_c$, $\tau_f$, $c$, and $\delta$ are in ms; gradient strength is in mT/m; $D_0$ is stored in m$^2$/ms; and the Capiglioni filter length $l_{cf}=\sqrt{D_0\tau_f}$ is displayed in $\mu$m. Normalized signal and contrast are dimensionless.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'Data-BIDS').is_dir() and (candidate / 'repos' / 'signal_analysis').is_dir():
            return candidate
    raise FileNotFoundError('Could not find the Project-Balseiro-Microstructure root.')


PROJECT_ROOT = find_project_root()
REPO_ROOT = PROJECT_ROOT / 'repos' / 'signal_analysis'
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# Reload local helpers so Run All also picks up code changes in an existing kernel.
import presentation_analysis.abstract_figures as _abstract_figures
import presentation_analysis as _presentation_analysis
importlib.reload(_abstract_figures)
importlib.reload(_presentation_analysis)

from presentation_analysis import (
    CC_ROIS,
    aggregate_alpha,
    build_resampled_contrasts,
    compare_with_reference,
    compare_with_reference_metrics,
    discover_analysis_runs,
    export_all_contrast_lcf_panels,
    export_all_signal_contrast_panels,
    fit_tc_pseudohuber,
    load_alpha_summaries,
    load_masters,
    master_qc_summary,
    merge_alpha_delta,
    normalize_alpha_to_internal_reference,
    plot_alpha_delta_scatter,
    plot_contrast_lcf_grid,
    plot_metric_comparison,
    plot_signal_contrast_example,
    save_figure_formats,
    summarize_contrast_shape,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_rows', 120)
pd.set_option('display.max_columns', 80)
PROJECT_ROOT

## 2. Configuration

The first lines under **Quick selection** are normally the only ones that need editing. Choose `DATASET = 'brains'` or `DATASET = 'phantoms'`, then set `SUBJECTS_OVERRIDE`, `ROI_OVERRIDE`, and `DIRECTION_OVERRIDE`. Use a Python list even for one value, for example `['20220610P']`, `['fiber1']`, or `['long']`; use `None` to include every value allowed by the dataset profile. `SHEETS_OVERRIDE` can isolate an acquisition when one subject has several sheets.

For phantoms, selecting exactly one known subject also selects its processed source automatically: `20220610P` uses `den_gr--manual`, whereas `20260706P` uses `den_gr-topup--manual`. `DWI_LEVEL_OVERRIDE` is therefore normally left as `None`. `EXAMPLE_*` and `EXAMPLE_TD_MS` change only the detailed example panels; they do not filter the complete analysis. `VARIANTS = None` discovers current and future ROI variants automatically.

The reference diffusivity $D_0$ is a physical prior used by the restricted-signal model and by the gradient-to-length transformation; it is not refitted independently for every ROI in this notebook. Consequently, brain and phantom values must remain explicit and should be reviewed before the phantom analysis is finalized. The $N=8$ and $N=4$ selection defines the two encoding waveforms whose difference is studied. `GRADIENT_COLUMN = 'g'` is the project default because it preserves the recorded, non-equally-spaced b-value sampling; use `g_thorsten` only for an explicit legacy sensitivity analysis.

`SAVE_OUTPUTS` controls persistent tables and presentation figures. Every saved figure is written in both PNG and PDF. `EXPORT_ALL_SIGNAL_CONTRAST_PANELS` controls the exhaustive Section 6 gallery. `EXPORT_ALL_CONTRAST_LCF_PANELS` creates a Figure-02-style image for every selected subject and ROI, with directions in columns and every available $T_D$ as a colored curve. Both are enabled by default.

In [ ]:
# -----------------------------------------------------------------------------
# QUICK SELECTION: edit this block first, then use Kernel > Restart & Run All.
# -----------------------------------------------------------------------------
DATASET = 'brains'  # 'brains' or 'phantoms'
#Examples: ['20220610P'], ['20260706P'], ['ADBN'], ['ADBN', 'ARVE', 'BRAIN', 'LUDG', 'MBBL', 'SNVN'], or None for all subjects
SUBJECTS_OVERRIDE = None
# Optional acquisition-sheet filter, useful when one brain subject has several scans
SHEETS_OVERRIDE = None            # e.g. ['20230623_BRAIN-4']
# Lists filter the complete analysis; None uses all defaults/available values
ROI_OVERRIDE = None               # phantom e.g. ['fiber1']; brain e.g. ['PostCC']
DIRECTION_OVERRIDE = None         # e.g. ['long'], ['tra'], or ['long', 'tra']

# One detailed signal/contrast panel (these settings do NOT filter the analysis)
EXAMPLE_SUBJECT_OVERRIDE = None   # None uses the selected subject when it is unique
EXAMPLE_ROI_OVERRIDE = None       # e.g. 'fiber1' or 'PostCC' (a string, not a list)
EXAMPLE_DIRECTION_OVERRIDE = None # e.g. 'long' or 'tra'
EXAMPLE_TD_MS = 143.4             # old phantom also has 142.5 ms; nearest value is used

# Dataset profiles: acquisition physics and default selections. Usually do not edit.
DATASET_PROFILES = {
    'brains': {
        'dwi_level': 'den_gr-topup',
        'd0_m2_ms': 3.2e-12,     # 0.0032 mm2/s
        'rois': list(CC_ROIS),
        'directions': ['long', 'tra'],
        'example_subject': 'ADBN',
        'example_roi': 'PostCC',
        'example_direction': 'long',
    },
    'phantoms': {
        'dwi_level': 'den_gr-topup',
        'd0_m2_ms': 2.3e-12,     # 0.0023 mm2/s
        'rois': None,               # None uses every ROI/sample found in the phantom master
        'directions': None,         # None prefers long/tra, then uses every available direction
        'example_subject': '20220610P',
        'example_roi': None,
        'example_direction': None,
    },
}

# Advanced data-source selection
SEQUENCE = 'ogse'
# Leave None: the notebook maps 20220610P -> den_gr and 20260706P -> den_gr-topup.
DWI_LEVEL_OVERRIDE = None          # force only when auditing a specific processing level
VARIANTS = None                   # brains e.g. ['plain'] or ['plain', 'erode1', 'sket1']
REFERENCE_VARIANT = None           # None prefers plain; phantoms use manual

# Contrast definition from the abstract
N_HIGH = 8
N_LOW = 4
# Use the gradient derived from each recorded b-value. This preserves non-equally-spaced
# acquisitions such as ADBN, ARVE, SNVN, and 20260706P. Set 'g_thorsten' only to
# reproduce the legacy equally-spaced-axis analysis.
GRADIENT_COLUMN = 'g'
APPLY_GRADIENT_CORRECTION = True
RESAMPLE_GRID_SIZE = 1000
SIGNAL_TC_BOUNDS_MS = (0.1, 10000.0)
RICIAN_C_BOUNDS = (0.0, 2.0)
SIGNAL_TC_MODE = 'separate'         # Flexible curve construction; not a physical two-tc interpretation

# Example-panel display limits
LCF_LIMITS_UM = (2.5, 11.0)  # Historical Capiglioni/abstract axis
TD_COLOR_TOLERANCE_MS = 8.0  # Includes both historical and current acquisition times

# Pseudo-Huber configuration used in the abstract analysis
EXCLUDE_TD_MS = (76.0,)
TD_RANGE_MS = (50.0, 250.0)
C_BOUNDS_MS = (0.0, 10.0)
DELTA_BOUNDS_MS = (1e-6, 10000.0)

# Transparent display-only QC. All fits remain in the exported audit tables.
MIN_PSEUDOHUBER_R2_FOR_SUMMARY = 0.50
EXCLUDE_DELTA_BOUNDARY_FROM_SUMMARY = True

# Exhaustive Section 6 export: one panel per subject/sheet/ROI/direction/TD combination.
EXPORT_ALL_SIGNAL_CONTRAST_PANELS = True
# Figure-02 gallery: one contrast-vs-filtered-length image per subject and ROI.
EXPORT_ALL_CONTRAST_LCF_PANELS = True
CLEAN_BATCH_PANEL_DIR = True       # Remove stale PNG/PDF files and the manifest inside each gallery
BATCH_PANEL_DPI = 180             # Lower than the presentation figures to control disk usage
BATCH_PROGRESS_EVERY = 25

SAVE_OUTPUTS = True

# Known phantom subjects use different processed derivatives.
PHANTOM_DWI_LEVEL_BY_SUBJECT = {
    '20220610P': 'den_gr',
    '20260706P': 'den_gr-topup',
}

def selection_list(value):
    if value is None:
        return None
    return [value] if isinstance(value, str) else list(value)

SELECTED_SUBJECTS = selection_list(SUBJECTS_OVERRIDE)
SELECTED_SHEETS = selection_list(SHEETS_OVERRIDE)
if DATASET not in DATASET_PROFILES:
    raise ValueError(f'Unknown DATASET={DATASET!r}; choose one of {list(DATASET_PROFILES)}.')
PROFILE = DATASET_PROFILES[DATASET]
if DWI_LEVEL_OVERRIDE is not None:
    DWI_LEVEL = DWI_LEVEL_OVERRIDE
elif DATASET == 'phantoms' and SELECTED_SUBJECTS is not None and len(SELECTED_SUBJECTS) == 1:
    DWI_LEVEL = PHANTOM_DWI_LEVEL_BY_SUBJECT.get(SELECTED_SUBJECTS[0], PROFILE['dwi_level'])
else:
    DWI_LEVEL = PROFILE['dwi_level']
D0_REFERENCE_M2_MS = PROFILE['d0_m2_ms']
OUTPUT_DIR = (
    REPO_ROOT / 'notebooks' / 'outputs' / 'dataset_metrics' / 'resample-ogse_rest_rician'
    / DATASET / DWI_LEVEL
)
OUTPUT_DIR

## 3. Discover variants and audit comparability

The analysis tag is parsed as `<DWI level>--<ROI variant>`. Newly created variants appear automatically when `VARIANTS` is `None`. Every table is loaded with its variant and dataset provenance attached before concatenation.

For the current corpus-callosum analysis, `plain` is the original labeled CC mask, `erode1` erodes the whole CC by one voxel and then restores the surviving original labels, and `sket1` is the CC skeleton dilated by one voxel and clipped to the original CC. Erosion reduces boundary partial volume but can delete thin subregions; the skeletonized band emphasizes the medial core but samples fewer and spatially different voxels. Therefore agreement across variants supports robustness to ROI geometry, whereas disagreement can reflect partial volume, SNR/voxel-count changes, or true spatial heterogeneity—it is not automatically an error in one method. Future variant names are treated as new provenance levels; their biological interpretation must come from the segmentation procedure that created them.

A fair segmentation comparison requires more than a similar total row count. Coverage is therefore compared on the complete logical signal key. `missing_rows` reports reference observations that cannot be matched in a candidate variant, while `extra_rows` reports the reverse. A missing ROI can be a legitimate consequence of erosion or skeletonization, but it reduces the paired sample and must remain visible. Exact or logical duplicates indicate an ingestion problem and should be resolved upstream.

Later group summaries use only matched subject/ROI/direction keys. No signal, fit, ROI, or subject is imputed to make variants appear balanced.

In [ ]:
runs = discover_analysis_runs(
    PROJECT_ROOT,
    dataset=DATASET,
    sequence=SEQUENCE,
    dwi_level=DWI_LEVEL,
)
available_variants = runs['variant'].tolist()
if VARIANTS is None:
    preferred_order = ['plain', 'erode1', 'sket1']
    selected_variants = [v for v in preferred_order if v in available_variants]
    selected_variants += [v for v in available_variants if v not in selected_variants]
else:
    selected_variants = list(VARIANTS)
if REFERENCE_VARIANT is None:
    REFERENCE_VARIANT = 'plain' if 'plain' in selected_variants else selected_variants[0]
if REFERENCE_VARIANT not in selected_variants:
    raise ValueError(f'Reference variant {REFERENCE_VARIANT!r} is not selected.')

display(runs[['dwi_level', 'variant', 'dataset', 'master_path']])
print('Selected variants:', selected_variants)

In [ ]:
masters = load_masters(runs, selected_variants)
alpha_raw = load_alpha_summaries(
    runs, selected_variants, reference_d0_m2_ms=D0_REFERENCE_M2_MS
)

# Filter complete tables, not only the example plot. Fail early on misspelled values.
for column, requested in [('subj', SELECTED_SUBJECTS), ('sheet', SELECTED_SHEETS)]:
    if requested is None:
        continue
    available = sorted(masters[column].dropna().astype(str).unique())
    missing = sorted(set(requested) - set(available))
    if missing:
        raise ValueError(
            f'{column} selection {missing} is unavailable for DATASET={DATASET!r}, '
            f'DWI_LEVEL={DWI_LEVEL!r}. Available values: {available}'
        )
    masters = masters[masters[column].astype(str).isin(requested)].copy()
    if column in alpha_raw.columns:
        alpha_raw = alpha_raw[alpha_raw[column].astype(str).isin(requested)].copy()
alpha_subject = aggregate_alpha(alpha_raw)

# Resolve analysis dimensions from the profile and the loaded master table.
rotated = masters.loc[masters['row_kind'].eq('signal_rotated')].copy()
available_subjects = sorted(rotated['subj'].dropna().astype(str).unique())
available_sheets = sorted(rotated['sheet'].dropna().astype(str).unique())
available_rois = sorted(rotated['roi'].dropna().astype(str).unique())
available_directions = sorted(rotated['direction'].dropna().astype(str).unique())
available_td_ms = sorted(pd.to_numeric(rotated['td_ms'], errors='coerce').dropna().unique())
requested_rois = ROI_OVERRIDE if ROI_OVERRIDE is not None else PROFILE['rois']
requested_directions = (
    DIRECTION_OVERRIDE if DIRECTION_OVERRIDE is not None else PROFILE['directions']
)
analysis_rois = (
    available_rois
    if requested_rois is None
    else [roi for roi in requested_rois if roi in available_rois]
)
if requested_directions is None:
    preferred_directions = [d for d in ['long', 'tra'] if d in available_directions]
    analysis_directions = preferred_directions or available_directions
else:
    analysis_directions = [d for d in requested_directions if d in available_directions]
if not analysis_rois or not analysis_directions:
    raise ValueError(
        f'No usable dimensions. Available ROIs={available_rois}; directions={available_directions}.'
    )

profile_example_subject = PROFILE['example_subject']
profile_example_roi = PROFILE['example_roi']
profile_example_direction = PROFILE['example_direction']
EXAMPLE_SUBJECT = (
    EXAMPLE_SUBJECT_OVERRIDE
    or (SELECTED_SUBJECTS[0] if SELECTED_SUBJECTS is not None and len(SELECTED_SUBJECTS) == 1 else None)
    or (profile_example_subject if profile_example_subject in available_subjects else None)
)
EXAMPLE_ROI = (
    EXAMPLE_ROI_OVERRIDE
    or (profile_example_roi if profile_example_roi in available_rois else None)
)
EXAMPLE_DIRECTION = (
    EXAMPLE_DIRECTION_OVERRIDE
    or (profile_example_direction if profile_example_direction in available_directions else None)
)

qc = master_qc_summary(masters)
coverage, missing_rows = compare_with_reference(
    masters,
    reference_variant=REFERENCE_VARIANT,
    row_kind='signal_rotated',
)
display(qc)
display(coverage)
print('Subjects:', available_subjects)
print('Sheets:', available_sheets)
print('Analysis ROIs:', analysis_rois)
print('Analysis directions:', analysis_directions)
print('Diffusion times [ms]:', available_td_ms)
print(
    'Requested example:',
    EXAMPLE_SUBJECT or 'auto',
    EXAMPLE_ROI or 'auto',
    EXAMPLE_DIRECTION or 'auto',
    EXAMPLE_TD_MS,
)

In [ ]:
if missing_rows.empty:
    print('Every selected variant has the same rotated-row coverage as the reference.')
else:
    missing_summary = (
        missing_rows.groupby(['variant', 'subj', 'roi', 'direction'], as_index=False)
        .size()
        .rename(columns={'size': 'missing_rows'})
    )
    display(missing_summary)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(coverage['variant'], coverage['coverage_percent'], color=plt.cm.Set2.colors[:len(coverage)])
ax.set_ylim(max(0, coverage['coverage_percent'].min() - 1), 100.1)
ax.set_ylabel(f'Coverage relative to {REFERENCE_VARIANT} [%]')
ax.set_title('Rotated-signal coverage audit')
for index, value in enumerate(coverage['coverage_percent']):
    ax.text(index, value, f'{value:.2f}%', ha='center', va='bottom')
fig.tight_layout()
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    save_figure_formats(fig, OUTPUT_DIR / '00_coverage_audit.png', dpi=300)
plt.show()

## 4. Gradient correction, signal fitting, resampling, and contrast

### 4.1 Why the gradient axis must be corrected first

The nominal gradient is not assumed to be the physical gradient experienced by the sample. Upstream calibration uses a reference ROI and computes

$$f_g=\sqrt{D_{0,\mathrm{NOGSE}}/\langle D_{0,\mathrm{monoexp}}\rangle}, \qquad g_{corr}=f_g g_{nominal}.$$

The factor may depend on acquisition, direction, and $N$. It is already stored in each master row as `grad_correction_factor`; this notebook applies it but does not estimate it again. Because the corrected $N=8$ and $N=4$ gradient samples generally do not coincide, subtracting rows with the same acquisition index would evaluate the two signals at different physical gradients and would not define $\Delta M(g)$ at a common $g$.

### 4.2 Restricted signal and normalized Rician floor

For each subject/sheet/ROI/direction/$T_D$ group, the two branches are fitted jointly. The clean restricted model can be summarized as

$$M_{rest}=M_0\exp[-\phi_{SE}-\phi_N+\phi_{cross}],$$

where the attenuation terms encode the finite correlation time $\tau_c$, waveform timing, $N$, corrected gradient, and fixed $D_0$. Here the waveform interval is $x=T_D/N$. Magnitude-MRI noise produces a non-zero floor, represented by

$$M(g,N,T_D)=\frac{\sqrt{M_{rest}(g,N,T_D)^2+C^2}}{\sqrt{1+C^2}}.$$

The denominator preserves $M(0)=1$ for normalized data. The fit has three free parameters: $\tau_{c,8}$, $\tau_{c,4}$, and one shared $C$. It fixes $M_0=1$ and the dataset-specific $D_0$. Sharing $C$ treats the two branches as measurements with a common magnitude-noise floor while allowing their effective correlation times to differ. The optimizer works in log space for positive $\tau_c$ values and reports branch-specific and combined $R^2$, RMSE, parameter errors, bounds, and status in `signal_pair_fit_summary.csv`.

### 4.3 Common-grid resampling

After fitting, both continuous models are evaluated on the same grid

$$g_j\in[0,\min(g_{8,max}^{corr},g_{4,max}^{corr})], \qquad j=1,\ldots,n_g,$$

and the contrast is finally computed as

$$\Delta M_{fit}(g_j)=M_{8,fit}(g_j)-M_{4,fit}(g_j).$$

Using only the overlap prevents the peak from being driven by extrapolation beyond one branch's measured range. The grid begins at zero, so both normalized fits equal one and $\Delta M_{fit}(0)=0$ by construction. `RESAMPLE_GRID_SIZE` controls numerical peak resolution; it does not add independent observations.

### 4.4 From the contrast peak to a filtered length and time

The sampled maximum gives $g_{peak}$. With $\gamma=267.5221900$ rad ms$^{-1}$ mT$^{-1}$ and the fixed $D_0$, the notebook uses the historical definition published by Capiglioni et al. (2021):

$$l_G=\left(\frac{D_0}{\gamma g}\right)^{1/3}, \quad l_D=\sqrt{D_0T_D}, \quad L_D=\frac{l_D}{l_G},$$

$$L_{cf}=\left(\frac{3}{2}\right)^{1/4}L_D^{-1/2}, \quad l_{cf}=L_{cf}l_G=\left(\frac{3D_0}{2\gamma^2g^2T_D}\right)^{1/4}, \quad \tau_{f,\mathrm{peak}}=\frac{l_{cf}^2}{D_0}.$$

This is the same physical-length convention used by the historical abstract figures. The alternative one-dimensional RMS displacement $\sqrt{2D_0\tau_f}$ is exactly $\sqrt{2}$ larger and is not used on these axes. These are model-derived coordinates, not independently measured histological sizes. Their interpretation inherits the assumptions of the restricted model, the chosen $D_0$, the gradient calibration, and the peak-selection rule.

In [ ]:
contrasts, contrast_fits = build_resampled_contrasts(
    masters,
    d0_m2_ms=D0_REFERENCE_M2_MS,
    n_high=N_HIGH,
    n_low=N_LOW,
    directions=analysis_directions,
    rois=analysis_rois,
    gradient_column=GRADIENT_COLUMN,
    apply_gradient_correction=APPLY_GRADIENT_CORRECTION,
    tc_bounds_ms=SIGNAL_TC_BOUNDS_MS,
    rician_c_bounds=RICIAN_C_BOUNDS,
    grid_size=RESAMPLE_GRID_SIZE,
    tc_mode=SIGNAL_TC_MODE,
)
contrast_shape = summarize_contrast_shape(contrasts)

contrast_fit_qc = (
    contrast_fits.groupby(['variant', 'ok'], as_index=False)
    .agg(curves=('td_ms', 'size'), median_r2=('r2', 'median'), median_rmse=('rmse', 'median'))
)
display(contrast_fit_qc)
display(
    contrast_fits.loc[contrast_fits['ok'].fillna(False)]
    .nsmallest(15, 'r2')[
        ['variant', 'subj', 'roi', 'direction', 'td_ms', 'r2_high', 'r2_low', 'r2', 'C']
    ]
)
print(f'Resampled contrast points: {len(contrasts):,}')
print(f'Signal pairs: {len(contrast_fits):,}; successful: {int(contrast_fits.ok.sum()):,}')

## 5. PGSE macroscopic coefficient and microscopic-to-macroscopic transition

### 5.1 Independent PGSE-derived constraint

The upstream alpha step uses tensor-projected diffusivities `D_proj` from rotated signal rows. At the configured b-value it averages the projected diffusivity across the available apparent diffusion times and normalizes it by a reference free diffusivity:

$$\alpha_{macro}=\frac{\langle D_{proj}\rangle}{D_{0,ref}}.$$

The summary spreadsheet carries the selected b-step/b-value, the number and range of apparent diffusion times, the mean and spread of $D_{proj}$, the reference $D_0$, and propagated uncertainty. If multiple acquisition sheets contribute the same subject/ROI/direction key, `aggregate_alpha` averages their estimates and propagates the reported independent errors. This quantity is determined before the NOGSE peak-transition fit and is held fixed there, which reduces parameter degeneracy.

### 5.2 Pseudo-Huber transition model (legacy sensitivity analysis)

For each variant, subject, ROI, and direction, the peak correlation times across diffusion times are fitted with

$$\tau_c(T_D)=c+\alpha\delta\left[\sqrt{1+(T_D/\delta)^2}-1\right],$$

where $\alpha$ is fixed to the PGSE-derived value and $c$ and $\delta$ are fitted. The parameter $c$ is the short-time intercept and $\delta$ controls the crossover scale. The limiting behavior makes the role of the model clear:

$$T_D\ll\delta:\quad \tau_c\approx c+\frac{\alpha T_D^2}{2\delta},$$

$$T_D\gg\delta:\quad \tau_c\approx c+\alpha(T_D-\delta).$$

Thus the curve is smooth and approximately quadratic at short times, then approaches a line with slope $\alpha$. At least three retained $T_D$ values are required for the two free parameters. Times in `EXCLUDE_TD_MS` are explicitly omitted to reproduce the chosen analysis definition; the retained interval is controlled by `TD_RANGE_MS`. With only four or five times, a high $R^2$ does not establish that the crossover was sampled: when $\delta>T_{D,max}$ the data occupy only the quadratic limb and $\delta$ is an extrapolated scale tied to the fixed $\alpha$.

Optimizer convergence is necessary but not sufficient. A solution at the $\delta$ upper bound usually means that the sampled time range did not identify the transition. The `presentation_ok` flag additionally applies the visible $R^2$ threshold and optional boundary exclusion only to summary plots. Every attempted fit, including failures and exclusions, remains in `pseudohuber_fit_summary.csv` and `pseudohuber_fit_exclusions.csv`.

In [ ]:
tc_data, tc_summary = fit_tc_pseudohuber(
    contrast_fits,
    alpha_subject,
    exclude_td_ms=EXCLUDE_TD_MS,
    td_range_ms=TD_RANGE_MS,
    c_bounds_ms=C_BOUNDS_MS,
    delta_bounds_ms=DELTA_BOUNDS_MS,
)
tc_summary['at_delta_boundary'] = (
    pd.to_numeric(tc_summary['delta_ms'], errors='coerce') >= 0.999 * DELTA_BOUNDS_MS[1]
)
td_support = tc_data.groupby(['variant', 'subj', 'roi', 'direction'], as_index=False).agg(
    td_min_observed_ms=('td_ms', 'min'), td_max_observed_ms=('td_ms', 'max')
)
tc_summary = tc_summary.merge(
    td_support, on=['variant', 'subj', 'roi', 'direction'], how='left', validate='one_to_one'
)
tc_summary['delta_to_td_max_ratio'] = tc_summary['delta_ms'] / tc_summary['td_max_observed_ms']
tc_summary['crossover_sampled'] = tc_summary['delta_ms'].between(
    tc_summary['td_min_observed_ms'], tc_summary['td_max_observed_ms'], inclusive='both'
)
tc_summary['presentation_ok'] = (
    tc_summary['ok'].fillna(False)
    & (pd.to_numeric(tc_summary['r2'], errors='coerce') >= MIN_PSEUDOHUBER_R2_FOR_SUMMARY)
)
if EXCLUDE_DELTA_BOUNDARY_FROM_SUMMARY:
    tc_summary['presentation_ok'] &= ~tc_summary['at_delta_boundary']

transition_qc = (
    tc_summary.groupby('variant', as_index=False)
    .agg(
        attempted=('ok', 'size'),
        converged=('ok', 'sum'),
        presentation_valid=('presentation_ok', 'sum'),
        median_r2=('r2', 'median'),
        boundary_fits=('at_delta_boundary', 'sum'),
        sampled_crossovers=('crossover_sampled', 'sum'),
        median_delta_to_td_max=('delta_to_td_max_ratio', 'median'),
    )
)
display(transition_qc)

fit_exclusions = tc_summary[~tc_summary['presentation_ok']].copy()
if not fit_exclusions.empty:
    display(fit_exclusions[['variant', 'subj', 'roi', 'direction', 'n_td', 'delta_ms', 'r2', 'at_delta_boundary', 'message']])

In [ ]:
metrics = merge_alpha_delta(alpha_subject, tc_summary)
metrics = metrics.merge(
    tc_summary[['variant', 'subj', 'roi', 'direction', 'presentation_ok', 'at_delta_boundary']],
    on=['variant', 'subj', 'roi', 'direction'],
    how='left',
    validate='one_to_one',
)
metrics.insert(0, 'dataset', DATASET)
metrics_for_plots = metrics[metrics['presentation_ok'].fillna(False)].copy()
display(metrics_for_plots.head())

## 8. Figure 3 analogue: regional $\delta$ and $\alpha$

The upper row shows the NOGSE-derived transition time $\delta$; the lower row shows the independent PGSE-derived $\alpha_{macro}$. Bars are arithmetic means across the valid subjects in each ROI/variant/direction cell and error bars are the standard error of that mean. Open markers retain every subject-level estimate, so the reader can distinguish a consistent variant shift from a bar driven by one subject.

All variants share axes within each direction. Only `presentation_ok` transition fits enter this figure, but the exclusion tables remain part of the output. A missing marker is missing/invalid data, not a zero. Because the valid subject count may differ after ROI loss or fit exclusion, paired differences in Section 10 are the correct basis for quantitative claims about segmentation effects.

In [ ]:
figure_3 = plot_metric_comparison(
    metrics_for_plots,
    variants=selected_variants,
    rois=analysis_rois,
    directions=analysis_directions,
    output_path=OUTPUT_DIR / '03_delta_alpha_by_region.png' if SAVE_OUTPUTS else None,
)
plt.show()

## 9. Figure 4 analogue: $\delta$ versus $\alpha$

Every point is one matched subject/ROI/direction estimate: x is the PGSE-derived $\alpha_{macro}$ and y is the NOGSE-derived transition time $\delta$. Color identifies the ROI and marker shape identifies the subject. Separate variant-by-direction panels prevent a change in ROI extraction from being hidden by overplotting.

This figure is descriptive: it shows clustering, outliers, subject consistency, and possible association between the two observables. It does not by itself establish correlation or causality. Any formal association should account for repeated measurements within subject, ROI, direction, and variant instead of treating all points as independent.

In [ ]:
figure_4 = plot_alpha_delta_scatter(
    metrics_for_plots,
    variants=selected_variants,
    rois=analysis_rois,
    directions=analysis_directions,
    output_path=OUTPUT_DIR / '04_alpha_vs_delta.png' if SAVE_OUTPUTS else None,
)
plt.show()

## 10. Paired variant effects and export

Only matched subject/ROI/direction combinations enter paired differences. For metric $x$, the exported definitions are

$$\Delta x=x_{variant}-x_{reference}, \qquad \Delta x[\%]=100\frac{x_{variant}-x_{reference}}{x_{reference}}.$$

This pairing removes between-subject and between-ROI level differences from the segmentation comparison. Missing ROIs or invalid fits are never imputed, and the matched-case count is reported beside every mean difference. Percent differences can become unstable when the reference is near zero, so absolute and relative effects should be read together. These summaries are descriptive and do not replace a repeated-measures statistical model if inferential claims are required.

In [ ]:
variant_differences = compare_with_reference_metrics(
    metrics_for_plots,
    reference_variant=REFERENCE_VARIANT,
)
variant_differences.insert(0, 'dataset', DATASET)
difference_summary = (
    variant_differences.groupby(['variant', 'direction'], as_index=False)
    .agg(
        matched_cases=('subj', 'size'),
        alpha_mean_difference=('alpha_macro_difference', 'mean'),
        alpha_mean_percent_difference=('alpha_macro_percent_difference', 'mean'),
        delta_mean_difference_ms=('delta_ms_difference', 'mean'),
        delta_mean_percent_difference=('delta_ms_percent_difference', 'mean'),
    )
)
difference_summary.insert(0, 'dataset', DATASET)
analysis_config = pd.DataFrame(
    [{
        'dataset': DATASET,
        'sequence': SEQUENCE,
        'dwi_level': DWI_LEVEL,
        'variants': ';'.join(selected_variants),
        'reference_variant': REFERENCE_VARIANT,
        'rois': ';'.join(analysis_rois),
        'directions': ';'.join(analysis_directions),
        'N_high': N_HIGH,
        'N_low': N_LOW,
        'D0_reference_m2_ms': D0_REFERENCE_M2_MS,
        'D0_reference_mm2_s': D0_REFERENCE_M2_MS * 1e9,
        'gradient_column': GRADIENT_COLUMN,
        'lcf_convention': 'capiglioni_2021',
        'lcf_definition': 'sqrt(D0 * tau_f)',
        'td_color_tolerance_ms': TD_COLOR_TOLERANCE_MS,
        'bvalue_column': 'bvalue_g',
        'gradient_correction': APPLY_GRADIENT_CORRECTION,
        'contrast_construction': 'restricted_signal_fits_shared_C_resampled_common_gradient',
        'resample_grid_size': RESAMPLE_GRID_SIZE,
        'signal_tc_bounds_ms': ';'.join(map(str, SIGNAL_TC_BOUNDS_MS)),
        'rician_C_bounds': ';'.join(map(str, RICIAN_C_BOUNDS)),
        'excluded_td_ms': ';'.join(map(str, EXCLUDE_TD_MS)),
        'td_min_ms': TD_RANGE_MS[0],
        'td_max_ms': TD_RANGE_MS[1],
        'minimum_pseudohuber_r2': MIN_PSEUDOHUBER_R2_FOR_SUMMARY,
        'exclude_delta_boundary': EXCLUDE_DELTA_BOUNDARY_FROM_SUMMARY,
        'export_all_signal_contrast_panels': EXPORT_ALL_SIGNAL_CONTRAST_PANELS,
        'export_all_contrast_lcf_panels': EXPORT_ALL_CONTRAST_LCF_PANELS,
        'clean_batch_panel_dir': CLEAN_BATCH_PANEL_DIR,
        'batch_panel_dpi': BATCH_PANEL_DPI,
        'figure_formats': 'png;pdf',
        'batch_panel_key': 'subj;sheet;roi;direction;td_ms',
        'contrast_lcf_panel_key': 'subj;roi',
    }]
)
display(difference_summary)

In [ ]:
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    analysis_config.to_csv(OUTPUT_DIR / 'analysis_config.csv', index=False)
    qc.to_csv(OUTPUT_DIR / 'master_qc.csv', index=False)
    coverage.to_csv(OUTPUT_DIR / 'coverage_vs_reference.csv', index=False)
    missing_rows.to_csv(OUTPUT_DIR / 'missing_master_rows.csv', index=False)
    contrasts.to_csv(OUTPUT_DIR / 'resampled_contrast.csv', index=False)
    contrast_fits.to_csv(OUTPUT_DIR / 'signal_pair_fit_summary.csv', index=False)
    contrast_shape.to_csv(OUTPUT_DIR / 'contrast_shape_metrics.csv', index=False)
    tc_data.rename(columns={'tc_peak_ms': 'tau_f_peak_ms', 'tc_peak_std_ms': 'tau_f_peak_std_ms'}).to_csv(
        OUTPUT_DIR / 'tau_f_peak_vs_td.csv', index=False
    )
    tc_data.to_csv(OUTPUT_DIR / 'tc_peak_vs_td.csv', index=False)
    tc_summary.to_csv(OUTPUT_DIR / 'pseudohuber_fit_summary.csv', index=False)
    fit_exclusions.to_csv(OUTPUT_DIR / 'pseudohuber_fit_exclusions.csv', index=False)
    metrics.to_csv(OUTPUT_DIR / 'alpha_delta_subject_summary.csv', index=False)
    variant_differences.to_csv(OUTPUT_DIR / 'variant_differences_vs_reference.csv', index=False)
    difference_summary.to_csv(OUTPUT_DIR / 'variant_difference_summary.csv', index=False)
    print(f'Figures and audit tables saved under: {OUTPUT_DIR}')

## Output map

`00_coverage_audit.*` (ROI-variant coverage vs. the reference variant), `03_delta_alpha_by_region.*` (regional delta/alpha bars), `04_alpha_vs_delta.*` (delta vs. alpha scatter), and every audit/fit table: `analysis_config.csv`, `master_qc.csv`, `coverage_vs_reference.csv`, `missing_master_rows.csv`, `resampled_contrast.csv`, `signal_pair_fit_summary.csv`, `contrast_shape_metrics.csv`, `tau_f_peak_vs_td.csv`, `tc_peak_vs_td.csv`, `pseudohuber_fit_summary.csv`, `pseudohuber_fit_exclusions.csv`, `alpha_delta_subject_summary.csv`, `variant_differences_vs_reference.csv`, `variant_difference_summary.csv`.

Before using these results in a presentation: report the coverage audit and any missing ROI/acquisition combinations; inspect pseudo-Huber exclusions and boundary fits (optimizer convergence alone is not biological validity); base segmentation-effect claims on `variant_differences_vs_reference.csv`, which reports the matched-case count.